#Preparación del entorno

In [0]:
%python
# CONFIGURACIÓN DEL ENTORNO
# Banking Analytics Lakehouse

# Widgets
dbutils.widgets.text("storage_account", "adlsprojectbank12")
dbutils.widgets.text("storage_credential", "credential-project")
dbutils.widgets.text("catalog_name", "banking_dev")

storage_account = dbutils.widgets.get("storage_account").strip()
storage_credential = dbutils.widgets.get("storage_credential").strip()
catalog_name = dbutils.widgets.get("catalog_name").strip()

In [0]:
%python
print("Configuración del entorno")
print(f"Storage Account     : {storage_account}")
print(f"Storage Credential  : {storage_credential}")
print(f"Catálogo            : {catalog_name}")

In [0]:
%python
# DEFINICIÓN DE RUTAS 

# Contenedores
containers = ["raw", "bronze", "silver", "golden", "metastore"]

storage_paths = { 
    container: (f"abfss://{container}@{storage_account}.dfs.core.windows.net/") for container in containers
}

In [0]:
%python
for container, path in storage_paths.items():
    print(f"{container:<12}: {path}")

In [0]:
%python
# STORAGE CREDENTIAL
credentials = spark.sql("SHOW STORAGE CREDENTIALS")

credential_exists = any(row["name"] == storage_credential
    for row in credentials.collect()
)

In [0]:
%python
if not credential_exists:
    raise Exception(
        f"No se ha encontrado ninguna Storage Credential: {storage_credential}"
    )

print(f"Storage Credential: {storage_credential}")

In [0]:
-- EXTERNAL LOCATIONS

-- Metastore
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-banking-metastore`
URL 'abfss://metastore@${storage_account}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential-project`)
COMMENT 'External Location para las tablas metastore del Data Lake';

-- Raw
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-banking-raw`
URL 'abfss://raw@${storage_account}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential-project`)
COMMENT 'External Location para las tablas raw del Data Lake';

-- Bronze
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-banking-bronze`
URL 'abfss://bronze@${storage_account}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential-project`)
COMMENT 'External Location para las tablas bronze del Data Lake';

-- Silver
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-banking-silver`
URL 'abfss://silver@${storage_account}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential-project`)
COMMENT 'External Location para las tablas silver del Data Lake';

-- Golden
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-banking-golden`
URL 'abfss://golden@${storage_account}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential-project`)
COMMENT 'External Location para las tablas golden del Data Lake';


In [0]:
-- Validar External Locations
DESCRIBE EXTERNAL LOCATION `exlt-banking-metastore`;

In [0]:
DESCRIBE EXTERNAL LOCATION `exlt-banking-raw`;

In [0]:
DESCRIBE EXTERNAL LOCATION `exlt-banking-bronze`;

In [0]:
DESCRIBE EXTERNAL LOCATION `exlt-banking-silver`;

In [0]:
DESCRIBE EXTERNAL LOCATION `exlt-banking-golden`;

In [0]:
-- CREAR EL CATÁLOGO
CREATE CATALOG IF NOT EXISTS banking_dev
MANAGED LOCATION 'abfss://metastore@${storage_account}.dfs.core.windows.net/catalogs/banking_dev/';

In [0]:
-- CREAR SCHEMAS
CREATE SCHEMA IF NOT EXISTS banking_dev.raw;

CREATE SCHEMA IF NOT EXISTS banking_dev.bronze;

CREATE SCHEMA IF NOT EXISTS banking_dev.silver;

CREATE SCHEMA IF NOT EXISTS banking_dev.golden;

##Creación de tablas Bronze

#### SBS

In [0]:
-- TABLAS BRONZE - SBS

CREATE TABLE IF NOT EXISTS banking_dev.bronze.sbs_creditos_sector_economico (
    fecha_corte DATE,
    concepto STRING,
    nivel_jerarquia INT,
    numero_deudores BIGINT,
    creditos_moneda_nacional_miles_soles DECIMAL(18,2),
    creditos_moneda_extranjera_miles_soles DECIMAL(18,2),
    total_creditos_directos_miles_soles DECIMAL(18,2),
    distribucion_creditos_directos_pct DECIMAL(10,4),
    ingestion_date TIMESTAMP
)
USING DELTA
LOCATION 'abfss://bronze@${storage_account}.dfs.core.windows.net/sbs_creditos_sector_economico';


CREATE TABLE IF NOT EXISTS banking_dev.bronze.sbs_depositos_mn (
    fecha_corte DATE,
    moneda STRING,
    empresa STRING,
    numero_cuentas BIGINT,
    saldo_anterior_miles_soles DECIMAL(18,2),
    abonos_miles_soles DECIMAL(18,2),
    intereses_capitalizados_miles_soles DECIMAL(18,2),
    retiros_cargos_miles_soles DECIMAL(18,2),
    saldo_final_miles_soles DECIMAL(18,2),
    ingestion_date TIMESTAMP
)
USING DELTA
LOCATION 'abfss://bronze@${storage_account}.dfs.core.windows.net/sbs_depositos_mn';


CREATE TABLE IF NOT EXISTS banking_dev.bronze.sbs_depositos_me (
    fecha_corte DATE,
    moneda STRING,
    empresa STRING,
    numero_cuentas BIGINT,
    saldo_anterior_miles_soles DECIMAL(18,2),
    abonos_miles_soles DECIMAL(18,2),
    intereses_capitalizados_miles_soles DECIMAL(18,2),
    retiros_cargos_miles_soles DECIMAL(18,2),
    saldo_final_miles_soles DECIMAL(18,2),
    ingestion_date TIMESTAMP
)
USING DELTA
LOCATION 'abfss://bronze@${storage_account}.dfs.core.windows.net/sbs_depositos_me';


CREATE TABLE IF NOT EXISTS banking_dev.bronze.sbs_morosidad (
    fecha_corte DATE,
    concepto_credito STRING,
    nivel_jerarquia INT,
    empresa_bancaria STRING,
    morosidad_pct DECIMAL(10,4),
    ingestion_date TIMESTAMP
)
USING DELTA
LOCATION 'abfss://bronze@${storage_account}.dfs.core.windows.net/sbs_morosidad';

#### FRAUD

In [0]:
-- TABLA BRONZE - FRAUD
CREATE TABLE IF NOT EXISTS banking_dev.bronze.fraud_transactions (
    Time DOUBLE,
    V1 DOUBLE,
    V2 DOUBLE,
    V3 DOUBLE,
    V4 DOUBLE,
    V5 DOUBLE,
    V6 DOUBLE,
    V7 DOUBLE,
    V8 DOUBLE,
    V9 DOUBLE,
    V10 DOUBLE,
    V11 DOUBLE,
    V12 DOUBLE,
    V13 DOUBLE,
    V14 DOUBLE,
    V15 DOUBLE,
    V16 DOUBLE,
    V17 DOUBLE,
    V18 DOUBLE,
    V19 DOUBLE,
    V20 DOUBLE,
    V21 DOUBLE,
    V22 DOUBLE,
    V23 DOUBLE,
    V24 DOUBLE,
    V25 DOUBLE,
    V26 DOUBLE,
    V27 DOUBLE,
    V28 DOUBLE,
    Amount DECIMAL(18,2),
    Class INT,
    ingestion_date TIMESTAMP
)
USING DELTA
LOCATION 'abfss://bronze@${storage_account}.dfs.core.windows.net/fraud_transactions';

#### BANKING

In [0]:
-- TABLAS BRONZE - BANKING

CREATE TABLE IF NOT EXISTS banking_dev.bronze.customers (
    customer_id BIGINT,
    document_type STRING,
    document_number STRING,
    first_name STRING,
    last_name STRING,
    birth_date DATE,
    gender STRING,
    city STRING,
    customer_segment STRING,
    registration_date DATE,
    customer_status STRING,
    ingestion_date TIMESTAMP
)
USING DELTA
LOCATION 'abfss://bronze@${storage_account}.dfs.core.windows.net/customers';


CREATE TABLE IF NOT EXISTS banking_dev.bronze.accounts (
    account_id BIGINT,
    customer_id BIGINT,
    account_type STRING,
    currency STRING,
    opening_date DATE,
    balance DECIMAL(18,2),
    account_status STRING,
    ingestion_date TIMESTAMP
)
USING DELTA
LOCATION 'abfss://bronze@${storage_account}.dfs.core.windows.net/accounts';


CREATE TABLE IF NOT EXISTS banking_dev.bronze.cards (
    card_id BIGINT,
    account_id BIGINT,
    card_type STRING,
    card_brand STRING,
    issue_date DATE,
    expiration_date DATE,
    card_status STRING,
    ingestion_date TIMESTAMP
)
USING DELTA
LOCATION 'abfss://bronze@${storage_account}.dfs.core.windows.net/cards';


CREATE TABLE IF NOT EXISTS banking_dev.bronze.transactions (
    transaction_id BIGINT,
    account_id BIGINT,
    card_id BIGINT,
    transaction_date TIMESTAMP,
    transaction_type STRING,
    channel STRING,
    merchant_category STRING,
    amount DECIMAL(18,2),
    currency STRING,
    transaction_status STRING,
    is_fraud INT,
    ingestion_date TIMESTAMP
)
USING DELTA
LOCATION 'abfss://bronze@${storage_account}.dfs.core.windows.net/transactions';


CREATE TABLE IF NOT EXISTS banking_dev.bronze.loans (
    loan_id BIGINT,
    customer_id BIGINT,
    loan_type STRING,
    principal_amount DECIMAL(18,2),
    interest_rate DECIMAL(8,4),
    term_months INT,
    disbursement_date DATE,
    due_date DATE,
    loan_status STRING,
    ingestion_date TIMESTAMP
)
USING DELTA
LOCATION 'abfss://bronze@${storage_account}.dfs.core.windows.net/loans';


CREATE TABLE IF NOT EXISTS banking_dev.bronze.payments (
    payment_id BIGINT,
    loan_id BIGINT,
    payment_date DATE,
    payment_amount DECIMAL(18,2),
    principal_paid DECIMAL(18,2),
    interest_paid DECIMAL(18,2),
    payment_status STRING,
    ingestion_date TIMESTAMP
)
USING DELTA
LOCATION 'abfss://bronze@${storage_account}.dfs.core.windows.net/payments';

In [0]:
-- Validar ruta
DESCRIBE DETAIL banking_dev.bronze.payments;

In [0]:
-- Mostrar tablas en esquemas
SHOW TABLES IN banking_dev.bronze;

##Creación de tablas Silver

In [0]:
-- TABLAS SILVER

--SBS
CREATE TABLE IF NOT EXISTS banking_dev.silver.sbs_creditos_sector_economico
LIKE banking_dev.bronze.sbs_creditos_sector_economico
LOCATION 'abfss://silver@${storage_account}.dfs.core.windows.net/sbs_creditos_sector_economico';

CREATE TABLE IF NOT EXISTS banking_dev.silver.sbs_depositos_mn
LIKE banking_dev.bronze.sbs_depositos_mn
LOCATION 'abfss://silver@${storage_account}.dfs.core.windows.net/sbs_depositos_mn';

CREATE TABLE IF NOT EXISTS banking_dev.silver.sbs_depositos_me
LIKE banking_dev.bronze.sbs_depositos_me
LOCATION 'abfss://silver@${storage_account}.dfs.core.windows.net/sbs_depositos_me';

CREATE TABLE IF NOT EXISTS banking_dev.silver.sbs_morosidad
LIKE banking_dev.bronze.sbs_morosidad
LOCATION 'abfss://silver@${storage_account}.dfs.core.windows.net/sbs_morosidad';

-- FRAUD
CREATE TABLE IF NOT EXISTS banking_dev.silver.fraud_transactions
LIKE banking_dev.bronze.fraud_transactions
LOCATION 'abfss://silver@${storage_account}.dfs.core.windows.net/fraud_transactions';

-- BANKING
CREATE TABLE IF NOT EXISTS banking_dev.silver.customers
LIKE banking_dev.bronze.customers
LOCATION 'abfss://silver@${storage_account}.dfs.core.windows.net/customers';

CREATE TABLE IF NOT EXISTS banking_dev.silver.accounts
LIKE banking_dev.bronze.accounts
LOCATION 'abfss://silver@${storage_account}.dfs.core.windows.net/accounts';

CREATE TABLE IF NOT EXISTS banking_dev.silver.cards
LIKE banking_dev.bronze.cards
LOCATION 'abfss://silver@${storage_account}.dfs.core.windows.net/cards';

CREATE TABLE IF NOT EXISTS banking_dev.silver.transactions
LIKE banking_dev.bronze.transactions
LOCATION 'abfss://silver@${storage_account}.dfs.core.windows.net/transactions';

CREATE TABLE IF NOT EXISTS banking_dev.silver.loans
LIKE banking_dev.bronze.loans
LOCATION 'abfss://silver@${storage_account}.dfs.core.windows.net/loans';

CREATE TABLE IF NOT EXISTS banking_dev.silver.payments
LIKE banking_dev.bronze.payments
LOCATION 'abfss://silver@${storage_account}.dfs.core.windows.net/payments';

In [0]:
-- Validar ruta
DESCRIBE DETAIL banking_dev.silver.payments;

In [0]:
-- Mostrar tablas en esquemas
SHOW TABLES IN banking_dev.silver;

## Creación de tablas Golden

In [0]:
-- TABLAS GOLDEN

CREATE TABLE IF NOT EXISTS banking_dev.golden.dim_customer (
    customer_key BIGINT,
    customer_id BIGINT,
    customer_segment STRING,
    city STRING,
    customer_status STRING
)
USING DELTA
LOCATION 'abfss://golden@${storage_account}.dfs.core.windows.net/dim_customer';


CREATE TABLE IF NOT EXISTS banking_dev.golden.dim_account (
    account_key BIGINT,
    account_id BIGINT,
    customer_id BIGINT,
    account_type STRING,
    currency STRING,
    account_status STRING
)
USING DELTA
LOCATION 'abfss://golden@${storage_account}.dfs.core.windows.net/dim_account';


CREATE TABLE IF NOT EXISTS banking_dev.golden.dim_card (
    card_key BIGINT,
    card_id BIGINT,
    account_id BIGINT,
    card_type STRING,
    card_brand STRING,
    card_status STRING
)
USING DELTA
LOCATION 'abfss://golden@${storage_account}.dfs.core.windows.net/dim_card';


CREATE TABLE IF NOT EXISTS banking_dev.golden.dim_date (
    date_key INT,
    full_date DATE,
    year INT,
    quarter INT,
    month INT,
    month_name STRING,
    day INT,
    day_of_week INT
)
USING DELTA
LOCATION 'abfss://golden@${storage_account}.dfs.core.windows.net/dim_date';


CREATE TABLE IF NOT EXISTS banking_dev.golden.fact_transactions (
    transaction_id BIGINT,
    date_key INT,
    customer_id BIGINT,
    account_id BIGINT,
    card_id BIGINT,
    amount DECIMAL(18,2),
    currency STRING,
    transaction_type STRING,
    channel STRING,
    transaction_status STRING,
    is_fraud INT
)
USING DELTA
LOCATION 'abfss://golden@${storage_account}.dfs.core.windows.net/fact_transactions';


CREATE TABLE IF NOT EXISTS banking_dev.golden.fact_loans (
    loan_id BIGINT,
    customer_id BIGINT,
    principal_amount DECIMAL(18,2),
    interest_rate DECIMAL(8,4),
    term_months INT,
    loan_status STRING
)
USING DELTA
LOCATION 'abfss://golden@${storage_account}.dfs.core.windows.net/fact_loans';


CREATE TABLE IF NOT EXISTS banking_dev.golden.fact_payments (
    payment_id BIGINT,
    loan_id BIGINT,
    payment_date DATE,
    payment_amount DECIMAL(18,2),
    principal_paid DECIMAL(18,2),
    interest_paid DECIMAL(18,2),
    payment_status STRING
)
USING DELTA
LOCATION 'abfss://golden@${storage_account}.dfs.core.windows.net/fact_payments';


CREATE TABLE IF NOT EXISTS banking_dev.golden.agg_sbs_creditos (
    fecha_corte DATE,
    concepto STRING,
    nivel_jerarquia INT,
    total_creditos_miles_soles DECIMAL(18,2),
    distribucion_pct DECIMAL(10,4)
)
USING DELTA
LOCATION 'abfss://golden@${storage_account}.dfs.core.windows.net/agg_sbs_creditos';


CREATE TABLE IF NOT EXISTS banking_dev.golden.agg_sbs_depositos (
    fecha_corte DATE,
    empresa STRING,
    moneda STRING,
    numero_cuentas BIGINT,
    saldo_final_miles_soles DECIMAL(18,2)
)
USING DELTA
LOCATION 'abfss://golden@${storage_account}.dfs.core.windows.net/agg_sbs_depositos';


CREATE TABLE IF NOT EXISTS banking_dev.golden.agg_sbs_morosidad (
    fecha_corte DATE,
    empresa_bancaria STRING,
    concepto_credito STRING,
    morosidad_pct DECIMAL(10,4)
)
USING DELTA
LOCATION 'abfss://golden@${storage_account}.dfs.core.windows.net/agg_sbs_morosidad';


CREATE TABLE IF NOT EXISTS banking_dev.golden.agg_fraud_daily (
    transaction_date DATE,
    total_transactions BIGINT,
    fraud_transactions BIGINT,
    legitimate_transactions BIGINT,
    fraud_amount DECIMAL(18,2),
    fraud_rate DECIMAL(10,4)
)
USING DELTA
LOCATION 'abfss://golden@${storage_account}.dfs.core.windows.net/agg_fraud_daily';

In [0]:
-- Validar ruta
DESCRIBE DETAIL banking_dev.golden.agg_fraud_daily;

In [0]:
-- Mostrar tablas en esquemas
SHOW TABLES IN banking_dev.golden;

In [0]:
--Comprobar que todas las tablas sean external
SELECT
    table_schema,
    table_name,
    table_type
FROM banking_dev.information_schema.tables
WHERE table_schema IN ('bronze', 'silver', 'golden')
ORDER BY table_schema, table_name;